# T2 — M2: Baseline rodando — Previsão do vencedor de uma corrida de F1
**Grupo:** Gustavo Salmazo da Silva · Gustavo Henrique Baliero Lima  
**Disciplina:** Redes Neurais e Aprendizado Profundo (EMA9824) — 2026/2

**O que este notebook entrega (M2):** baseline medido nas mesmas condições do modelo final
(mesma divisão temporal, mesma métrica, mesmo pré-processamento), com **curva de treino e validação**,
**valor da métrica** e **tempo de execução**.

> O conjunto de **teste (2022–2024) NÃO é avaliado aqui**: ele só é consultado uma única vez, no M3.

Como executar: `Runtime > Run all`. CPU basta (~1 min no total).

## 0. Ambiente, semente e configuração

In [ ]:
import time, json, platform, zipfile, io, os, random
T_START = time.perf_counter()          # wall-clock for the whole notebook

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
import matplotlib.pyplot as plt
import requests

# ---- Experiment configuration (declared once, reused everywhere) ----
SEEDS      = [0, 1, 2, 3, 4]    # number of repeated runs = 5
EPOCHS     = 40                 # fixed budget; best-validation checkpoint is kept
BATCH_SIZE = 256
LR         = 0.05
FEATURES   = ["grid_adj", "cons_pts_frac"]   # baseline uses ONLY these two (see proposal, field 3)
TRAIN_END  = 2018               # train: 1950-2018
VAL_YEARS  = (2019, 2021)       # validation: 2019-2021
TEST_START = 2022               # test: 2022-2024 (locked until M3)
DATA_DIR   = "f1_data"
KAGGLE_URL = "https://www.kaggle.com/api/v1/datasets/download/rohanrao/formula-1-world-championship-1950-2020"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Python", platform.python_version(), "| torch", torch.__version__,
      "| pandas", pd.__version__, "| numpy", np.__version__, "| device:", device)

## 1. Dados (obtidos da fonte original, sem arquivos no repositório)
Dataset público do Kaggle **rohanrao/formula-1-world-championship-1950-2020** (corridas de 1950 a 2024, licença aberta
declarada na página do dataset). As tabelas usadas: `results`, `races`.  
*Nota de rastreabilidade:* o link da proposta (M1) aponta para `julianbloise/winners-formula-1-1950-to-2025`, que traz apenas
o **vencedor** de cada GP e não permite montar a grade completa de largada. Por isso usamos o dataset de resultados completos
(mesma família de tabelas — `races`, `results`, `qualifying`, `drivers`, `constructors` — já usada no nosso projeto de ML).

In [ ]:
def download_f1(data_dir=DATA_DIR):
    """Download the public CSV bundle from Kaggle (no credentials needed) and cache it."""
    if os.path.exists(f"{data_dir}/results.csv"):
        return
    os.makedirs(data_dir, exist_ok=True)
    try:
        r = requests.get(KAGGLE_URL, timeout=120)
        r.raise_for_status()
        zipfile.ZipFile(io.BytesIO(r.content)).extractall(data_dir)
    except Exception as e:                                   # fallback: kagglehub
        print("Direct download failed (", e, ") -> trying kagglehub")
        import kagglehub
        path = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")
        for f in os.listdir(path):
            os.replace(os.path.join(path, f), os.path.join(data_dir, f))

t0 = time.perf_counter()
download_f1()
results = pd.read_csv(f"{DATA_DIR}/results.csv", na_values="\\N")
races   = pd.read_csv(f"{DATA_DIR}/races.csv",   na_values="\\N")
T_DATA = time.perf_counter() - t0
print(f"results: {results.shape} | races: {races.shape} | years {races.year.min()}-{races.year.max()} | {T_DATA:.1f}s")

## 2. Pré-processamento (só informação **pré-corrida**, sem vazamento)
| Variável | Definição | Por que não vaza |
|---|---|---|
| `win` (alvo) | `positionOrder == 1` | é o rótulo |
| `grid_adj` | posição de largada; `grid = 0` (box/sem dado) vira *último + 1* da corrida | largada é conhecida antes da prova |
| `cons_pts_frac` | pontos acumulados da equipe **antes** desta corrida ÷ pontos do líder entre os inscritos (0 na 1ª etapa) | acumulado deslocado 1 etapa (`shift`); normalizar pelo líder torna comparável entre eras com pontuações diferentes |

Padronização (z-score) usa **média e desvio calculados só no treino**.

In [ ]:
df = (results[["raceId", "driverId", "constructorId", "grid", "positionOrder", "points"]]
      .merge(races[["raceId", "year", "round"]], on="raceId")
      .sort_values(["year", "round", "raceId"]).reset_index(drop=True))

df["win"] = (df["positionOrder"] == 1).astype(np.float32)

# grid == 0 means pit-lane start / missing -> treat as last on the grid
gmax = df.groupby("raceId")["grid"].transform("max")
df["grid_adj"] = np.where(df["grid"] == 0, gmax + 1, df["grid"]).astype(np.float32)

# constructor points accumulated BEFORE each race (current race excluded via shift)
cons = (df.groupby(["year", "round", "constructorId"], as_index=False)["points"].sum()
          .sort_values(["constructorId", "year", "round"]))
cons["pts_before"] = (cons.groupby(["year", "constructorId"])["points"].cumsum() - cons["points"])
df = df.merge(cons[["year", "round", "constructorId", "pts_before"]],
              on=["year", "round", "constructorId"])

# normalise by the leader among the entrants of that race (era-independent)
leader = df.groupby("raceId")["pts_before"].transform("max")
df["cons_pts_frac"] = np.where(leader > 0, df["pts_before"] / leader, 0.0).astype(np.float32)

# ---- strictly temporal split ----
train = df[df.year <= TRAIN_END].copy()
val   = df[(df.year >= VAL_YEARS[0]) & (df.year <= VAL_YEARS[1])].copy()
test  = df[df.year >= TEST_START]            # NOT used in M2 (only counted)

# sanity checks
# 3 early races (1951 FRA, 1956 ARG, 1957 GBR) have a shared-drive co-winner -> 2 rows with win=1; all others have exactly 1
wins_per_race = df.groupby("raceId")["win"].sum()
assert wins_per_race.ge(1).all() and wins_per_race.le(2).all() and (wins_per_race == 2).sum() == 3
# a pick counts as a hit if the chosen driver is any of the race winners
assert train.year.max() < val.year.min() <= val.year.max() < TEST_START

split_info = pd.DataFrame({
    "linhas":  [len(train), len(val), len(test)],
    "corridas":[train.raceId.nunique(), val.raceId.nunique(), test.raceId.nunique()],
    "anos":    [f"{train.year.min()}-{train.year.max()}", f"{val.year.min()}-{val.year.max()}",
                f"{test.year.min()}-{test.year.max()}"],
    "uso":     ["treino", "validação", "TESTE (bloqueado até o M3)"]}, index=["train", "val", "test"])
display(split_info)
print("taxa de vitória por linha (treino):", round(train.win.mean(), 4), "(~1 vencedor para ~20 pilotos)")

# z-score with TRAIN statistics only
mu, sd = train[FEATURES].mean(), train[FEATURES].std().replace(0, 1)
def to_tensor(d):
    X = ((d[FEATURES] - mu) / sd).values.astype(np.float32)
    return torch.tensor(X), torch.tensor(d["win"].values.astype(np.float32))
Xtr, ytr = to_tensor(train); Xva, yva = to_tensor(val)

## 3. Métrica
Como ~95 % das linhas são "não vence", a acurácia por linha seria enganosa (prever "ninguém vence" já dá ~95 %).
Usamos **acurácia de vencedor por corrida (top-1)**: em cada corrida o modelo escolhe **um** piloto (maior pontuação) e
acerta se ele realmente venceu. É a mesma métrica que será usada no M3 (modelo final) e no teste.
Como métrica auxiliar para a curva, a **entropia cruzada binária (BCE)** por linha.

In [ ]:
def race_top1_acc(frame, score):
    """Race-level accuracy: pick the highest-score driver in each race; hit if he actually won."""
    tmp = frame[["raceId", "win"]].copy()
    tmp["score"] = np.asarray(score)
    pick = tmp.loc[tmp.groupby("raceId")["score"].idxmax()]
    return float(pick["win"].mean())

# ---- Baseline A: naive heuristic "pole position wins" (lowest grid_adj) ----
pole_train = race_top1_acc(train, -train["grid_adj"].values)
pole_val   = race_top1_acc(val,   -val["grid_adj"].values)

# ---- Reference: random pick among the entrants (expected accuracy = mean(1/n_cars)) ----
rand_val = float((1 / val.groupby("raceId")["win"].size()).mean())

print(f"Heurística pole  | treino: {pole_train:.3f} | validação: {pole_val:.3f}  ({val.raceId.nunique()} corridas)")
print(f"Aleatório        | validação: {rand_val:.3f}")

## 4. Baseline B: regressão logística (= 1 neurônio sigmoide) em PyTorch
`nn.Linear(2, 1)` + `BCEWithLogitsLoss` é exatamente uma regressão logística treinada por gradiente (Adam).
Cinco sementes (inicialização + ordem dos minibatches). **Parada antecipada / checkpoint:** a cada época avaliamos a
validação e guardamos os pesos da melhor época (menor BCE de validação) em disco; esse é o estado usado nas métricas finais.

In [ ]:
def run_seed(seed):
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    g = torch.Generator().manual_seed(seed)                   # controls minibatch shuffling
    model = nn.Linear(len(FEATURES), 1)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    loss_fn = nn.BCEWithLogitsLoss()
    hist = {"train_loss": [], "val_loss": [], "val_acc": [], "train_acc": []}
    best = {"val_loss": float("inf"), "epoch": -1}
    n = len(Xtr)

    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(n, generator=g)
        for i in range(0, n, BATCH_SIZE):
            idx = perm[i:i + BATCH_SIZE]
            opt.zero_grad()
            loss_fn(model(Xtr[idx]).squeeze(1), ytr[idx]).backward()
            opt.step()

        model.eval()
        with torch.no_grad():                                  # full-set metrics, no gradient
            s_tr, s_va = model(Xtr).squeeze(1), model(Xva).squeeze(1)
            tl, vl = loss_fn(s_tr, ytr).item(), loss_fn(s_va, yva).item()
        hist["train_loss"].append(tl); hist["val_loss"].append(vl)
        hist["train_acc"].append(race_top1_acc(train, s_tr.numpy()))
        hist["val_acc"].append(race_top1_acc(val, s_va.numpy()))

        if vl < best["val_loss"]:                              # checkpoint on best validation
            best = {"val_loss": vl, "epoch": ep}
            torch.save(model.state_dict(), f"best_seed{seed}.pt")

    hist["best_epoch"] = best["epoch"]
    hist["best_val_loss"] = best["val_loss"]
    hist["best_val_acc"] = hist["val_acc"][best["epoch"]]      # metric at the checkpoint that is kept
    hist["best_train_acc"] = hist["train_acc"][best["epoch"]]
    return hist

t0 = time.perf_counter()
runs = {s: run_seed(s) for s in SEEDS}
T_TRAIN = time.perf_counter() - t0

summary = pd.DataFrame({s: {"melhor época": r["best_epoch"] + 1, "BCE val": r["best_val_loss"],
                            "acc. corrida (treino)": r["best_train_acc"],
                            "acc. corrida (val)": r["best_val_acc"]} for s, r in runs.items()}).T
summary.index.name = "semente"
display(summary.round(4))

acc_val = np.array([r["best_val_acc"] for r in runs.values()])
acc_tr  = np.array([r["best_train_acc"] for r in runs.values()])
bce_val = np.array([r["best_val_loss"] for r in runs.values()])
print(f"Regressão logística | acc. por corrida (val): {acc_val.mean():.3f} ± {acc_val.std(ddof=1):.3f} (n={len(SEEDS)} sementes)")
print(f"Tempo de treino ({len(SEEDS)} sementes x {EPOCHS} épocas): {T_TRAIN:.1f}s")

## 5. Figura do M2 — curva de treino e validação

In [ ]:
E = np.arange(1, EPOCHS + 1)
stack = lambda k: np.array([r[k] for r in runs.values()])
C_TR, C_VA, C_POLE, C_RND = "#2a6fdb", "#e0701a", "#444444", "#888888"

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# (a) loss curves: mean +- std across seeds
for key, c, lab in [("train_loss", C_TR, "treino"), ("val_loss", C_VA, "validação")]:
    m, s = stack(key).mean(0), stack(key).std(0, ddof=1)
    ax[0].plot(E, m, color=c, label=lab); ax[0].fill_between(E, m - s, m + s, color=c, alpha=0.2)
be = int(np.median([r["best_epoch"] for r in runs.values()])) + 1
ax[0].axvline(be, color="k", ls=":", lw=1, label=f"melhor época (mediana = {be})")
ax[0].set_xlabel("época"); ax[0].set_ylabel("entropia cruzada binária (BCE)")
ax[0].set_title("(a) Perda — regressão logística"); ax[0].legend()

# (b) race-level accuracy on validation vs. heuristics
m, s = stack("val_acc").mean(0), stack("val_acc").std(0, ddof=1)
ax[1].plot(E, m, color=C_VA, label="regressão logística (val)")
ax[1].fill_between(E, m - s, m + s, color=C_VA, alpha=0.2)
ax[1].axhline(pole_val, color=C_POLE, ls="--", label=f"heurística pole ({pole_val:.3f})")
ax[1].axhline(rand_val, color=C_RND, ls=":", label=f"aleatório ({rand_val:.3f})")
ax[1].set_xlabel("época"); ax[1].set_ylabel("acurácia de vencedor por corrida")
ax[1].set_title("(b) Validação 2019–2021 (60 corridas)"); ax[1].legend(loc="lower right"); ax[1].set_ylim(0, 1)

fig.suptitle("Baseline F1 — treino 1950–2018 | validação 2019–2021 | média ± desvio, 5 sementes", y=1.02)
fig.tight_layout()
fig.savefig("m2_baseline.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. Resumo do M2 (copiar para o relatório) e arquivo de métricas

In [ ]:
T_TOTAL = time.perf_counter() - T_START
n_val_races = val.raceId.nunique()
metrics = {
    "split": {"train": f"1950-{TRAIN_END}", "val": f"{VAL_YEARS[0]}-{VAL_YEARS[1]}", "test": "2022-2024 (not evaluated in M2)"},
    "n_races": {"train": int(train.raceId.nunique()), "val": int(n_val_races), "test": int(test.raceId.nunique())},
    "n_rows":  {"train": int(len(train)), "val": int(len(val))},
    "seeds": SEEDS, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LR, "features": FEATURES,
    "pole_acc": {"train": pole_train, "val": pole_val},
    "random_acc_val": rand_val,
    "logreg_acc_val_mean": float(acc_val.mean()), "logreg_acc_val_std": float(acc_val.std(ddof=1)),
    "logreg_acc_train_mean": float(acc_tr.mean()),
    "logreg_bce_val_mean": float(bce_val.mean()), "logreg_bce_val_std": float(bce_val.std(ddof=1)),
    "best_epochs": [r["best_epoch"] + 1 for r in runs.values()],
    "time_s": {"data": T_DATA, "train": T_TRAIN, "total": T_TOTAL},
    "env": {"python": platform.python_version(), "torch": torch.__version__, "pandas": pd.__version__,
            "device": str(device), "platform": platform.platform()},
}
json.dump(metrics, open("m2_metrics.json", "w"), indent=2)

diff = acc_val.mean() - pole_val
if abs(diff) <= acc_val.std(ddof=1):
    verdict = "diferença dentro da dispersão entre sementes"
else:
    verdict = "acima da heurística" if diff > 0 else "abaixo da heurística"
print(f"""
================ M2 — RESUMO ================
Ambiente: {metrics['env']['platform']} | Python {metrics['env']['python']} | torch {metrics['env']['torch']} | device {device}
Dados: {metrics['n_rows']['train']} linhas / {metrics['n_races']['train']} corridas (treino), {metrics['n_rows']['val']} / {n_val_races} (validação)
Pole (val):               {pole_val:.3f}
Regressão logística (val): {acc_val.mean():.3f} ± {acc_val.std(ddof=1):.3f}   -> {verdict} (Δ = {diff:+.3f})
Aleatório (val):          {rand_val:.3f}
BCE val (logística):      {bce_val.mean():.4f} ± {bce_val.std(ddof=1):.4f}
Melhores épocas por semente: {metrics['best_epochs']}
Tempo: dados {T_DATA:.1f}s | treino {T_TRAIN:.1f}s | total {T_TOTAL:.1f}s
=============================================
""")

## 7. Próximos passos (M3)
1. **Melhoria:** MLP (e/ou XGBoost) com features pré-corrida mais ricas — médias móveis (últimas 5 corridas) de piloto e equipe,
   histórico no circuito, tempo de qualificação — **um fator por vez**, 3+ sementes, mesma métrica.
2. **Condições comparadas (M1, campo 5):** treino completo vs. era híbrida (2014+); com/sem features de circuito/clima; corridas com muitos DNF.
3. **Teste 2022–2024:** consultado **uma única vez**, ao final.
4. **Cuidado herdado do projeto de ML:** o conjunto de features anterior incluía `laps`, que só é conhecida **após** a corrida (vazamento);
   no T2 não será usada.